In [1]:
# !pip install --q --upgrade trl
!pip install -q --upgrade accelerate
!pip install -q autoawq
!pip install -q bitsandbytes

In [2]:
from datasets import load_dataset
from datasets import concatenate_datasets
import re
from transformers import BitsAndBytesConfig, TrainingArguments, AutoProcessor, AutoModelForImageTextToText

In [3]:
!pip install --upgrade transformers

In [4]:
model_name = "Qwen/Qwen3.5-2B"

In [5]:
torch.cuda.empty_cache()

NameError: name 'torch' is not defined

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForImageTextToText.from_pretrained("Qwen/Qwen3.5-2B",quantization_config=bnb_config)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

In [7]:
model.eval()

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear4bit(in_features=1024, out_features=3072, bias=True)
            (proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear4bit(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
   

In [8]:
import torch
print("Allocated:", torch.cuda.memory_allocated()/1024**3, "GB")
print("Reserved:", torch.cuda.memory_reserved()/1024**3, "GB")

Allocated: 1.7852263450622559 GB
Reserved: 1.828125 GB


In [9]:
# ds_1 = load_dataset("chillies/IELTS-writing-task-2-evaluation")
ds_2 = load_dataset("chillies/IELTS_evaluations")


In [10]:
# ds_train = concatenate_datasets({ds_1["train"], ds_2["train"]})
# ds_test = concatenate_datasets({ds_1["test"], ds_2["test"]})
ds_train = concatenate_datasets({ ds_2["train"]})
ds_test = concatenate_datasets({ ds_2["test"]})
print(ds_train)
print(ds_test)

Dataset({
    features: ['prompt', 'essay', 'evaluation', 'band'],
    num_rows: 9912
})
Dataset({
    features: ['prompt', 'essay', 'evaluation', 'band'],
    num_rows: 495
})


In [13]:
prompt_template_with_input = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}
"""

instruction_prompt ="""You are an IELTS Writing Task 2 examiner.
 Evaluate the essay below in detail according to the four official IELTS Writing criteria: Task Achievement, Coherence and Cohesion, Lexical Resource, and Grammatical Range and Accuracy.
 For each criterion, provide specific comments, examples, and a suggested band score.
 Then, provide an Overall Band Score and general feedback including Strengths, Areas for Improvement, and Suggestions for Enhancement.
 """

print(prompt_template_with_input.format(instruction = instruction_prompt,
                                        input = f"Essay Topic:\n {ds_train[0]["prompt"]}\n\nEssay:\n{ds_train[0]["essay"]} ",
                                        output =  f"Evaluation:\n{data["evaluation"]}\n\nOverall Band Score :\n{data["band"]}"
                                        ))

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an IELTS Writing Task 2 examiner.
 Evaluate the essay below in detail according to the four official IELTS Writing criteria: Task Achievement, Coherence and Cohesion, Lexical Resource, and Grammatical Range and Accuracy.
 For each criterion, provide specific comments, examples, and a suggested band score.
 Then, provide an Overall Band Score and general feedback including Strengths, Areas for Improvement, and Suggestions for Enhancement.
 

### Input:
Essay Topic:
 Young people who commit crimes should be treated in the same way as adults who commit crimes.

To what extent do you agree or disagree?

Essay:
Deciding to choose among the potential ways of punishing young people who commit crimes continues to be a controversial issue for the societies and the governments. It is argued by some that these peopl

In [14]:
process_data = []
for data in ds_train :
  formatted_input = prompt_template_with_input.format(instruction = instruction_prompt,
                                                      input = f"Essay Topic:\n {data["prompt"]}\n\nEssay:\n{data["essay"]} ",
                                                      output =  f"Evaluation:\n{data["evaluation"]}\n\nOverall Band Score :\n{data["band"]}"
                                                      )
  process_data.append({"text": formatted_input, "output" : f"Evaluation:\n{data["evaluation"]}\n\nOverall Band Score :\n{data["band"]}"})


In [15]:
process_data[1]

{'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nYou are an IELTS Writing Task 2 examiner.\n Evaluate the essay below in detail according to the four official IELTS Writing criteria: Task Achievement, Coherence and Cohesion, Lexical Resource, and Grammatical Range and Accuracy.\n For each criterion, provide specific comments, examples, and a suggested band score.\n Then, provide an Overall Band Score and general feedback including Strengths, Areas for Improvement, and Suggestions for Enhancement.\n \n\n### Input:\nEssay Topic:\n Young people who commit crimes should be treated in the same way as adults who commit crimes. To what extend do you agree or disagree?\n\nEssay:\nIn this modern era, youngsters who commit offences should be punished in  similar ways as mature people. I do think that younger criminals should have behaved less strictly than o

In [16]:
!pip install -q peft
!pip install -q trl


In [17]:
from peft import LoraConfig , get_peft_model , prepare_model_for_kbit_training
from trl import SFTTrainer
from sklearn.model_selection import train_test_split


In [18]:
from datasets import Dataset
train_indices, val_indices = train_test_split(range(len(process_data)), test_size=0.1, random_state=42)
process_dataset = Dataset.from_list(process_data)
train_dataset = process_dataset.select(train_indices)
val_dataset = process_dataset.select(val_indices)

In [20]:
lora_config = LoraConfig (
  r=8 , # Rank
  lora_alpha =32 , # Alpha parameter
  target_modules =[ # Target attention matrices
  "q_proj",
  "v_proj",
  ],
  lora_dropout =  0.05,
  bias = "none",
  task_type = "CAUSAL_LM"
)


In [21]:
import transformers, trl, peft, accelerate, bitsandbytes

print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)


transformers: 5.5.4
trl: 1.1.0
peft: 0.18.1
accelerate: 1.13.0
bitsandbytes: 0.49.2


In [22]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 417,792 || all params: 2,213,659,456 || trainable%: 0.0189


In [23]:
training_args = TrainingArguments(
  output_dir = "./content/output",
  num_train_epochs = 2 ,
  per_device_train_batch_size = 4,
  gradient_accumulation_steps= 4,
  learning_rate = 1e-4,
  fp16=True,
  save_total_limit = 3,
  logging_steps = 1,
  optim="paged_adamw_8bit"
)

In [24]:
trainer = SFTTrainer(
  model=model,
  train_dataset=train_dataset,
  args=training_args,
  eval_dataset=val_dataset,
)
trainer.train()

Adding EOS to train dataset:   0%|          | 0/8920 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8920 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/992 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/992 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 5.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.30 GiB is allocated by PyTorch, and 122.54 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)